# Test Metrics Summary

This notebook reads a `test_metrics_over_all.csv` file and prints mean/std per metric, overall and by body part.
Use the utility functions below to load a CSV and display results for multiple models in the same notebook.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

BASE_METRICS = ["MAE", "MSE", "PSNR", "SSIM"]


def _metric_columns(base_metrics=BASE_METRICS):
    unmasked = [f"{m}_unmasked" for m in base_metrics]
    masked = [f"{m}_masked" for m in base_metrics]
    return unmasked + masked


def _sanitize_metric_columns(df: pd.DataFrame, metrics) -> pd.DataFrame:
    """Convert +/-inf to NaN so mean/std are computed on finite values only."""
    df = df.copy()
    for col in metrics:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            df[col] = df[col].replace([np.inf, -np.inf], np.nan)
    return df


def _ensure_metric_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize metrics to *_unmasked and *_masked columns."""
    df = df.copy()
    suffixed_cols = _metric_columns()
    if all(col in df.columns for col in suffixed_cols):
        return df

    # Backward compatibility with old CSVs that only had MAE/MSE/PSNR/SSIM.
    if all(col in df.columns for col in BASE_METRICS):
        for metric in BASE_METRICS:
            df[f"{metric}_unmasked"] = pd.to_numeric(df[metric], errors="coerce")
            df[f"{metric}_masked"] = pd.to_numeric(df[metric], errors="coerce")
        return df

    missing = [c for c in suffixed_cols if c not in df.columns]
    raise ValueError(f"Could not find expected metric columns. Missing: {missing}")


def load_metrics_csv(csv_path: Path) -> pd.DataFrame:
    if not csv_path.exists():
        raise FileNotFoundError(f"CSV not found: {csv_path}")

    df = pd.read_csv(csv_path)

    if "file_name" not in df.columns:
        df = df.iloc[:, 1:]

    if "file_name" not in df.columns:
        raise ValueError("Could not find 'file_name' column after dropping index column.")

    df = _ensure_metric_columns(df)
    df = _sanitize_metric_columns(df, _metric_columns())

    if "mask_voxels" in df.columns:
        df["mask_voxels"] = pd.to_numeric(df["mask_voxels"], errors="coerce")

    return df


def load_volume_metrics_csv(volume_csv_path: Path) -> pd.DataFrame:
    if not volume_csv_path.exists():
        raise FileNotFoundError(f"CSV not found: {volume_csv_path}")

    df = pd.read_csv(volume_csv_path)

    if "volume_id" not in df.columns:
        df = df.iloc[:, 1:]

    if "volume_id" not in df.columns:
        raise ValueError("Could not find 'volume_id' column after dropping index column.")

    df = _ensure_metric_columns(df)
    df = _sanitize_metric_columns(df, _metric_columns())

    if "mask_voxels" in df.columns:
        df["mask_voxels"] = pd.to_numeric(df["mask_voxels"], errors="coerce")
    if "slice_count" in df.columns:
        df["slice_count"] = pd.to_numeric(df["slice_count"], errors="coerce")

    return df


def _fmt(v: float) -> str:
    return "n/a" if pd.isna(v) else f"{v:.6g}"


def overall_mean_std(df: pd.DataFrame, metrics) -> pd.DataFrame:
    overall_mean = df[metrics].mean()
    overall_std = df[metrics].std()
    return pd.DataFrame({"mean +- std": [f"{_fmt(m)} +- {_fmt(s)}" for m, s in zip(overall_mean, overall_std)]}, index=metrics)


def paired_mean_std(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for metric in BASE_METRICS:
        u_col = f"{metric}_unmasked"
        m_col = f"{metric}_masked"
        u_mean, u_std = df[u_col].mean(), df[u_col].std()
        m_mean, m_std = df[m_col].mean(), df[m_col].std()
        rows.append({
            "metric": metric,
            "unmasked (mean +- std)": f"{_fmt(u_mean)} +- {_fmt(u_std)}",
            "masked (mean +- std)": f"{_fmt(m_mean)} +- {_fmt(m_std)}",
        })
    return pd.DataFrame(rows).set_index("metric")


def add_body_part(df: pd.DataFrame, col: str = "file_name") -> pd.DataFrame:
    # Body part grouping derived from filename prefix (e.g., AB_*, HN_*)
    df = df.copy()
    df["body_part"] = df[col].astype(str).str.split("_").str[0]
    return df


def mean_std_by_body_part(df: pd.DataFrame) -> pd.DataFrame:
    grouped = df.groupby("body_part")[_metric_columns()].agg(["mean", "std"])

    formatted = pd.DataFrame(index=grouped.index)
    for metric in BASE_METRICS:
        for suffix in ["unmasked", "masked"]:
            col = f"{metric}_{suffix}"
            mean_col = (col, "mean")
            std_col = (col, "std")
            formatted[col] = grouped.apply(
                lambda row: f"{_fmt(row[mean_col])} +- {_fmt(row[std_col])}",
                axis=1,
            )
    return formatted


def _print_non_finite_counts(df: pd.DataFrame, title: str) -> None:
    issues = {}
    for metric in _metric_columns():
        non_finite = (~np.isfinite(df[metric])).sum()
        if non_finite:
            issues[metric] = int(non_finite)
    if issues:
        counts = ", ".join([f"{m}: {c}" for m, c in issues.items()])
        print(f"{title} | non-finite values ignored in stats -> {counts}")


def show_metrics_for_csv(csv_path: Path) -> None:
    df = load_metrics_csv(csv_path)

    _print_non_finite_counts(df, "Slice metrics")
    print("Overall metrics (slice-based):")
    display(paired_mean_std(df))

    df_bp = add_body_part(df, col="file_name")
    print("Metrics by body part (slice-based):")
    display(mean_std_by_body_part(df_bp))

    volume_csv_path = csv_path.parent / "test_metrics_over_volume.csv"
    if volume_csv_path.exists():
        vol_df = load_volume_metrics_csv(volume_csv_path)
        _print_non_finite_counts(vol_df, "Volume metrics")
        print("Overall metrics (volume-based):")
        display(paired_mean_std(vol_df))

        vol_df_bp = add_body_part(vol_df, col="volume_id")
        print("Metrics by body part (volume-based):")
        display(mean_std_by_body_part(vol_df_bp))
    else:
        print(f"Volume metrics CSV not found: {volume_csv_path}")
        print("Run: python preprocessing/preprocessing_synthrad/110compute_volume_metrics.py")



In [ ]:
# Per-slice metrics and influence for a single volume

def show_volume_slice_influence(df: pd.DataFrame, volume=None) -> None:
    # volume can be a volume_id string (e.g., 'AB_1ABA009') or an index into the unique volumes list
    if "file_name" not in df.columns:
        raise ValueError("df must contain a 'file_name' column.")

    def _volume_key(name: str) -> str:
        base = name
        if base.endswith(".nii.gz"):
            base = base[:-7]
        elif base.endswith(".nii"):
            base = base[:-4]
        # Use the part before the last '-' as volume id
        return base.rsplit("-", 1)[0]

    def _slice_key(name: str) -> str:
        base = name
        if base.endswith(".nii.gz"):
            base = base[:-7]
        elif base.endswith(".nii"):
            base = base[:-4]
        parts = base.rsplit("-", 1)
        return parts[1] if len(parts) == 2 else ""

    df = _ensure_metric_columns(df.copy())
    df["volume_id"] = df["file_name"].astype(str).map(_volume_key)
    df["slice_id"] = df["file_name"].astype(str).map(_slice_key)

    volumes = sorted(df["volume_id"].dropna().unique())
    if not volumes:
        print("No volumes found in df.")
        return

    if volume is None:
        volume_id = volumes[0]
    elif isinstance(volume, int):
        if volume < 0 or volume >= len(volumes):
            raise IndexError(f"volume index out of range: {volume} (0..{len(volumes)-1})")
        volume_id = volumes[volume]
    else:
        volume_id = str(volume)
        if volume_id not in volumes:
            raise ValueError(f"volume_id not found: {volume_id}")

    vol_df = df[df["volume_id"] == volume_id].copy()
    if vol_df.empty:
        print(f"No slices found for volume {volume_id}.")
        return

    metrics_cols = _metric_columns()
    vol_df[metrics_cols] = vol_df[metrics_cols].apply(pd.to_numeric, errors="coerce")

    if "mask_voxels" not in vol_df.columns:
        print("No mask_voxels column found; cannot compute masked slice influence.")
        display(vol_df[["file_name", "slice_id", *metrics_cols]])
        return

    vol_df["mask_voxels"] = pd.to_numeric(vol_df["mask_voxels"], errors="coerce").fillna(0)
    total_mask = vol_df["mask_voxels"].sum()
    if total_mask == 0:
        print("mask_voxels sum to 0 for this volume; cannot compute masked influence.")
        display(vol_df[["file_name", "slice_id", *metrics_cols, "mask_voxels"]])
        return

    vol_df["weight_pct"] = (vol_df["mask_voxels"] / total_mask) * 100

    # Volume-weighted masked metrics + unmasked means for this volume
    summary_rows = []
    for metric in BASE_METRICS:
        u_col = f"{metric}_unmasked"
        m_col = f"{metric}_masked"
        weights = vol_df["mask_voxels"] / total_mask
        summary_rows.append({
            "metric": metric,
            "unmasked_mean": vol_df[u_col].mean(),
            "masked_weighted_mean": (vol_df[m_col] * weights).sum(),
        })

    print(f"Volume: {volume_id} (slices={len(vol_df)}, total_mask={int(total_mask)})")
    display(pd.DataFrame(summary_rows).set_index("metric"))

    # Per-slice influence table
    show_cols = ["slice_id", *metrics_cols, "mask_voxels", "weight_pct", "file_name"]
    display(vol_df.sort_values("slice_id")[show_cols])

# Example usage (pick a volume index or id)
# show_volume_slice_influence(df, volume=0)
# show_volume_slice_influence(df, volume="AB_1ABA009")



# CUT


## Abdomen


In [ ]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cut_synthrad_abdomen_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [ ]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cut_synthrad_abdomen_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,36.3499 +- 21.2158,118.834 +- 58.5118
MSE,15002.4 +- 19484.7,49345.6 +- 55555.8
PSNR,26.5364 +- 3.67713,21.5034 +- 4.87443
SSIM,0.839716 +- 0.0440047,0.62013 +- 0.0722848


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,36.3499 +- 21.2158,118.834 +- 58.5118,15002.4 +- 19484.7,49345.6 +- 55555.8,26.5364 +- 3.67713,21.5034 +- 4.87443,0.839716 +- 0.0440047,0.62013 +- 0.0722848


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,36.0711 +- 10.1691,119.202 +- 29.6626
MSE,14519.2 +- 9013.13,48702.6 +- 29054.5
PSNR,26.7229 +- 1.44671,21.3842 +- 1.76962
SSIM,0.839158 +- 0.0268656,0.619675 +- 0.0309146


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,36.0711 +- 10.1691,119.202 +- 29.6626,14519.2 +- 9013.13,48702.6 +- 29054.5,26.7229 +- 1.44671,21.3842 +- 1.76962,0.839158 +- 0.0268656,0.619675 +- 0.0309146


## Brain


In [ ]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cut_synthrad_brain_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [ ]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cut_synthrad_brain_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,45.1682 +- 37.769,180.717 +- 149.697
MSE,30126.1 +- 42258.3,127270 +- 167272
PSNR,23.5588 +- 3.15695,17.8059 +- 3.67163
SSIM,0.898355 +- 0.0516616,0.73428 +- 0.156331


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
brain,45.1682 +- 37.769,180.717 +- 149.697,30126.1 +- 42258.3,127270 +- 167272,23.5588 +- 3.15695,17.8059 +- 3.67163,0.898355 +- 0.0516616,0.73428 +- 0.156331


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,44.9316 +- 10.336,150.888 +- 28.8315
MSE,29868.2 +- 11352.1,99815.3 +- 33933.8
PSNR,23.5911 +- 1.08483,18.5715 +- 0.988191
SSIM,0.898735 +- 0.0165647,0.756722 +- 0.0374324


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
brain,44.9316 +- 10.336,150.888 +- 28.8315,29868.2 +- 11352.1,99815.3 +- 33933.8,23.5911 +- 1.08483,18.5715 +- 0.988191,0.898735 +- 0.0165647,0.756722 +- 0.0374324


## Pelvis


In [ ]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cut_synthrad_pelvis_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [ ]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cut_synthrad_pelvis_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,29.4105 +- 43.234,97.3549 +- 66.2945
MSE,14109.5 +- 43514.6,45236.2 +- 69407
PSNR,27.1093 +- 2.55843,21.6155 +- 2.54583
SSIM,0.890386 +- 0.0467566,0.750303 +- 0.0881869


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
pelvis,29.4105 +- 43.234,97.3549 +- 66.2945,14109.5 +- 43514.6,45236.2 +- 69407,27.1093 +- 2.55843,21.6155 +- 2.54583,0.890386 +- 0.0467566,0.750303 +- 0.0881869


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,29.0182 +- 7.61076,95.9329 +- 16.9005
MSE,13783.7 +- 7021.23,43961.4 +- 14716.4
PSNR,27.1088 +- 1.12139,21.6941 +- 1.18487
SSIM,0.891334 +- 0.0162652,0.750814 +- 0.0344813


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
pelvis,29.0182 +- 7.61076,95.9329 +- 16.9005,13783.7 +- 7021.23,43961.4 +- 14716.4,27.1088 +- 1.12139,21.6941 +- 1.18487,0.891334 +- 0.0162652,0.750814 +- 0.0344813


## Thorax


In [ ]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cut_synthrad_TH_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


FileNotFoundError: CSV not found: /local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cut_synthrad_TH_final/test_50/test_metrics_over_all.csv

In [ ]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cut_synthrad_TH_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,92.6543 +- 301.843,190.791 +- 283.109
MSE,114274 +- 528585,157556 +- 507206
PSNR,24.7419 +- 4.82629,19.0985 +- 3.95581
SSIM,0.829226 +- 0.156578,0.642609 +- 0.137325


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
TH,92.6543 +- 301.843,190.791 +- 283.109,114274 +- 528585,157556 +- 507206,24.7419 +- 4.82629,19.0985 +- 3.95581,0.829226 +- 0.156578,0.642609 +- 0.137325


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,97.2005 +- 52.0121,155.654 +- 38.3489
MSE,121678 +- 86020.4,93822.1 +- 46385
PSNR,24.6512 +- 1.97691,19.5478 +- 1.77913
SSIM,0.826183 +- 0.0409675,0.657615 +- 0.0486513


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
TH,97.2005 +- 52.0121,155.654 +- 38.3489,121678 +- 86020.4,93822.1 +- 46385,24.6512 +- 1.97691,19.5478 +- 1.77913,0.826183 +- 0.0409675,0.657615 +- 0.0486513


## Head and Neck


In [ ]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cut_synthrad_HN_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [ ]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cut_synthrad_HN_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,186.381 +- 454.977,280.539 +- 428.507
MSE,722.735 +- 2573.17,1092.07 +- 7665.41
PSNR,20.9014 +- 5.82993,17.1293 +- 4.89265
SSIM,0.748259 +- 0.195435,0.600698 +- 0.175513


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
HN,186.381 +- 454.977,280.539 +- 428.507,722.735 +- 2573.17,1092.07 +- 7665.41,20.9014 +- 5.82993,17.1293 +- 4.89265,0.748259 +- 0.195435,0.600698 +- 0.175513


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,220.038 +- 136.947,310.551 +- 165.448
MSE,605.811 +- 515.959,896.636 +- 1543.59
PSNR,20.2255 +- 3.05245,16.6319 +- 3.02403
SSIM,0.734978 +- 0.0628657,0.582894 +- 0.0933291


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
HN,220.038 +- 136.947,310.551 +- 165.448,605.811 +- 515.959,896.636 +- 1543.59,20.2255 +- 3.05245,16.6319 +- 3.02403,0.734978 +- 0.0628657,0.582894 +- 0.0933291


## Fullbody


In [ ]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cut_synthrad_allregions_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [ ]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cut_synthrad_allregions_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,84.5819 +- 286.71,181.106 +- 281.175
MSE,960.134 +- 1226.11,3235.65 +- 3931.77
PSNR,24.405 +- 4.76696,19.1321 +- 4.20613
SSIM,0.847666 +- 0.135624,0.677079 +- 0.147906


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,98.1944 +- 347.922,171.829 +- 325.449,1318.92 +- 1032.67,4681.05 +- 4522.03,25.4908 +- 5.069,20.4053 +- 4.1859,0.824772 +- 0.151086,0.616842 +- 0.123862
HN,178.83 +- 452.805,262.587 +- 429.41,1667.01 +- 2454.65,4131.62 +- 6086.67,21.6981 +- 6.11348,17.9261 +- 5.14483,0.7664 +- 0.201552,0.63673 +- 0.186562
TH,101.626 +- 356.437,196.057 +- 337.632,994.072 +- 676.603,4267.31 +- 4766.46,24.8516 +- 5.06037,19.1991 +- 4.20674,0.832808 +- 0.156796,0.649473 +- 0.138453
brain,45.7914 +- 56.1777,176.887 +- 146.08,458.572 +- 199.344,1626.01 +- 858.324,23.5676 +- 3.26589,17.7934 +- 3.56651,0.896768 +- 0.050685,0.726881 +- 0.144994
pelvis,32.4928 +- 86.9599,106.113 +- 85.2909,777.534 +- 196.074,2803.77 +- 545.858,26.9345 +- 2.82816,21.2778 +- 2.73855,0.876766 +- 0.0486144,0.71202 +- 0.0937712


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,105.197 +- 105.685,176.534 +- 110.666
MSE,1094.62 +- 551.624,3416.89 +- 1321.13
PSNR,24.1838 +- 2.99864,19.384 +- 2.5202
SSIM,0.831768 +- 0.0676652,0.668699 +- 0.0748214


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,104.991 +- 95.4016,151.875 +- 61.4835,1389.27 +- 521.759,4540.75 +- 1299.49,25.509 +- 1.96212,20.7277 +- 1.49729,0.821261 +- 0.0504105,0.622502 +- 0.040727
HN,212.629 +- 137.153,294.985 +- 166.617,1712.13 +- 331.692,4287.15 +- 879.153,20.9235 +- 3.35572,17.2681 +- 3.2801,0.75193 +- 0.0660216,0.615758 +- 0.0965054
TH,106.909 +- 61.001,154.143 +- 42.4856,1007.54 +- 316.376,3723.31 +- 533.498,24.7472 +- 2.05258,19.6732 +- 1.89889,0.829651 +- 0.0393947,0.663964 +- 0.0455308
brain,45.6432 +- 10.4622,151.156 +- 27.5659,458.081 +- 58.0883,1558.92 +- 144.748,23.5907 +- 1.09345,18.4232 +- 0.966604,0.896871 +- 0.0134715,0.743843 +- 0.0268678
pelvis,31.9384 +- 12.2583,104.188 +- 19.174,768.831 +- 119.854,2780.95 +- 278.347,26.8729 +- 1.33419,21.298 +- 1.32992,0.876867 +- 0.0170696,0.709191 +- 0.0294098


# Pix2Pix


## Abdomen


In [ ]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/pix2pix_synthrad_abdomen_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [ ]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/pix2pix_synthrad_abdomen_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,36.2474 +- 21.3825,119.202 +- 61.9141
MSE,17072.8 +- 19294.3,55805.8 +- 56475.3
PSNR,25.9672 +- 4.08361,20.7568 +- 3.87527
SSIM,0.853325 +- 0.0393709,0.64022 +- 0.0837517


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,36.2474 +- 21.3825,119.202 +- 61.9141,17072.8 +- 19294.3,55805.8 +- 56475.3,25.9672 +- 4.08361,20.7568 +- 3.87527,0.853325 +- 0.0393709,0.64022 +- 0.0837517


Volume metrics CSV not found: /local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/pix2pix_synthrad_abdomen_final/test_50/test_metrics_over_volume.csv
Run: python preprocessing/preprocessing_synthrad/110compute_volume_metrics.py


## Brain


In [ ]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/pix2pix_synthrad_brain_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [ ]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/pix2pix_synthrad_brain_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,42.6191 +- 27.4481,156.961 +- 107.299
MSE,26391.6 +- 30479.1,99841.3 +- 121842
PSNR,24.1845 +- 3.35824,18.5677 +- 3.49392
SSIM,0.896493 +- 0.0521847,0.733482 +- 0.157675


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
brain,42.6191 +- 27.4481,156.961 +- 107.299,26391.6 +- 30479.1,99841.3 +- 121842,24.1845 +- 3.35824,18.5677 +- 3.49392,0.896493 +- 0.0521847,0.733482 +- 0.157675


Volume metrics CSV not found: /local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/pix2pix_synthrad_brain_final/test_50/test_metrics_over_volume.csv
Run: python preprocessing/preprocessing_synthrad/110compute_volume_metrics.py


## Pelvis


In [ ]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/pix2pix_synthrad_pelvis_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [ ]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/pix2pix_synthrad_pelvis_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,24.6755 +- 11.5469,85.2721 +- 32.5598
MSE,10313.8 +- 11245.3,33748.9 +- 28424.4
PSNR,27.6283 +- 2.29843,22.3809 +- 2.28194
SSIM,0.898945 +- 0.0252115,0.764254 +- 0.0837324


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
pelvis,24.6755 +- 11.5469,85.2721 +- 32.5598,10313.8 +- 11245.3,33748.9 +- 28424.4,27.6283 +- 2.29843,22.3809 +- 2.28194,0.898945 +- 0.0252115,0.764254 +- 0.0837324


Volume metrics CSV not found: /local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/pix2pix_synthrad_pelvis_final/test_50/test_metrics_over_volume.csv
Run: python preprocessing/preprocessing_synthrad/110compute_volume_metrics.py


## Thorax


In [ ]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/pix2pix_synthrad_thorax_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [ ]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/pix2pix_synthrad_thorax_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,35.0383 +- 13.3968,132.355 +- 50.9654
MSE,16228.2 +- 10294.1,62421.5 +- 42890
PSNR,26.1015 +- 5.1639,20.1768 +- 4.45819
SSIM,0.865309 +- 0.0334103,0.671369 +- 0.0824053


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
TH,35.0383 +- 13.3968,132.355 +- 50.9654,16228.2 +- 10294.1,62421.5 +- 42890,26.1015 +- 5.1639,20.1768 +- 4.45819,0.865309 +- 0.0334103,0.671369 +- 0.0824053


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,35.2119 +- 7.48374,133.533 +- 29.0677
MSE,16294.8 +- 6150.56,62406 +- 25546.7
PSNR,26.1565 +- 1.48609,19.8262 +- 1.668
SSIM,0.864924 +- 0.0218119,0.66673 +- 0.0474435


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
TH,35.2119 +- 7.48374,133.533 +- 29.0677,16294.8 +- 6150.56,62406 +- 25546.7,26.1565 +- 1.48609,19.8262 +- 1.668,0.864924 +- 0.0218119,0.66673 +- 0.0474435


## Head and Neck


In [ ]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/pix2pix_synthrad_headneck_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [ ]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/pix2pix_synthrad_headneck_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,59.688 +- 51.9265,145.59 +- 123.111
MSE,950.266 +- 516.082,2246.9 +- 760.33
PSNR,24.253 +- 5.53424,20.2209 +- 5.21463
SSIM,0.820472 +- 0.0877957,0.690596 +- 0.121756


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
HN,59.688 +- 51.9265,145.59 +- 123.111,950.266 +- 516.082,2246.9 +- 760.33,24.253 +- 5.53424,20.2209 +- 5.21463,0.820472 +- 0.0877957,0.690596 +- 0.121756


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,64.0622 +- 28.7262,161.135 +- 77.5774
MSE,924.464 +- 155.618,2281.34 +- 232.672
PSNR,24.0186 +- 1.78303,19.8059 +- 1.94035
SSIM,0.815366 +- 0.0392334,0.667739 +- 0.0826987


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
HN,64.0622 +- 28.7262,161.135 +- 77.5774,924.464 +- 155.618,2281.34 +- 232.672,24.0186 +- 1.78303,19.8059 +- 1.94035,0.815366 +- 0.0392334,0.667739 +- 0.0826987


## Fullbody


In [ ]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/pix2pix_synthrad_allregion_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [ ]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/pix2pix_synthrad_allregion_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Slice metrics | non-finite values ignored in stats -> PSNR_masked: 2
Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,42.3422 +- 34.6463,137.841 +- 98.1394
MSE,742.845 +- 365.468,2436.64 +- 888.481
PSNR,25.0512 +- 4.60595,19.8127 +- 4.08642
SSIM,0.866211 +- 0.0633841,0.697125 +- 0.127683


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,35.7609 +- 21.3131,114.784 +- 59.5105,909.13 +- 291.923,3012.93 +- 796.87,26.0537 +- 4.60045,20.9343 +- 3.72383,0.856352 +- 0.039229,0.651934 +- 0.0768345
HN,66.1476 +- 62.3865,160.199 +- 148.144,1010.57 +- 562.478,2407.03 +- 928.044,23.7718 +- 5.77799,19.7662 +- 5.44028,0.811456 +- 0.0937256,0.674224 +- 0.127973
TH,37.0716 +- 14.3373,138.831 +- 57.1808,801.24 +- 260.856,2986.25 +- 701.868,25.6981 +- 5.93247,19.7705 +- 4.23982,0.859999 +- 0.0359638,0.665227 +- 0.0921901
brain,43.8641 +- 25.9867,159.73 +- 97.8801,517.577 +- 217.082,1855.2 +- 786.982,23.8769 +- 3.36234,18.2584 +- 3.42243,0.891781 +- 0.0577551,0.722838 +- 0.166238
pelvis,28.3824 +- 18.9812,100.329 +- 80.4681,662.208 +- 158.507,2396.35 +- 528.313,26.6892 +- 2.31971,21.4683 +- 2.52391,0.889723 +- 0.0300076,0.746238 +- 0.0855961


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,44.481 +- 24.4606,137.221 +- 59.9666
MSE,785.352 +- 220.524,2520.03 +- 550.055
PSNR,25.1524 +- 1.89368,19.9706 +- 1.86181
SSIM,0.858761 +- 0.0407015,0.687322 +- 0.0685984


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,35.0436 +- 10.6388,113.526 +- 34.5038,908.685 +- 175.444,3014.75 +- 366.143,26.3034 +- 1.54279,20.9644 +- 1.8187,0.858139 +- 0.0255154,0.653966 +- 0.0492277
HN,71.6321 +- 36.3344,180.559 +- 98.383,990.59 +- 166.044,2445.48 +- 300.629,23.5458 +- 1.79954,19.3112 +- 2.00788,0.80685 +- 0.038914,0.650528 +- 0.083066
TH,37.2465 +- 8.04886,139.504 +- 33.724,805.09 +- 157.448,3044.28 +- 283.545,25.7557 +- 1.52057,19.4741 +- 1.73451,0.859768 +- 0.0220713,0.660293 +- 0.0545041
brain,43.8175 +- 6.71016,144.308 +- 16.6779,515.133 +- 64.5071,1726.51 +- 162.532,23.8832 +- 0.912779,18.7546 +- 0.865869,0.89186 +- 0.0133934,0.738155 +- 0.0336808
pelvis,28.6316 +- 5.48226,98.58 +- 16.4061,661.653 +- 82.5421,2385.71 +- 240.435,26.631 +- 0.90845,21.495 +- 0.924415,0.888723 +- 0.0157249,0.741845 +- 0.0302484


# CycleGAN


## Abdomen


In [ ]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cyclegan_abdomen_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [ ]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cyclegan_abdomen_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,62.5837 +- 37.6788,178.264 +- 112.808
MSE,1326.06 +- 471.628,4390.17 +- 1341.66
PSNR,23.0299 +- 5.50007,18.8181 +- 4.49552
SSIM,0.801874 +- 0.0470071,0.521518 +- 0.110381


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,62.5837 +- 37.6788,178.264 +- 112.808,1326.06 +- 471.628,4390.17 +- 1341.66,23.0299 +- 5.50007,18.8181 +- 4.49552,0.801874 +- 0.0470071,0.521518 +- 0.110381


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,59.4257 +- 20.2198,174.472 +- 54.3367
MSE,1346.59 +- 272.374,4467.66 +- 597.608
PSNR,23.4928 +- 2.89715,18.8446 +- 2.09964
SSIM,0.805771 +- 0.0307603,0.527928 +- 0.0819799


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,59.4257 +- 20.2198,174.472 +- 54.3367,1346.59 +- 272.374,4467.66 +- 597.608,23.4928 +- 2.89715,18.8446 +- 2.09964,0.805771 +- 0.0307603,0.527928 +- 0.0819799


## Brain


In [ ]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cyclegan_brain_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [ ]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cyclegan_brain_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,88.7851 +- 45.4188,301.096 +- 126.009
MSE,535.347 +- 224.47,1812.13 +- 737.072
PSNR,19.3218 +- 3.68977,13.6109 +- 2.68268
SSIM,0.80957 +- 0.0788591,0.517178 +- 0.167235


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
brain,88.7851 +- 45.4188,301.096 +- 126.009,535.347 +- 224.47,1812.13 +- 737.072,19.3218 +- 3.68977,13.6109 +- 2.68268,0.80957 +- 0.0788591,0.517178 +- 0.167235


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,89.1496 +- 21.9109,298.628 +- 58.686
MSE,534.995 +- 78.6223,1761.77 +- 188.454
PSNR,19.2954 +- 1.44903,13.5041 +- 1.42003
SSIM,0.808978 +- 0.0364361,0.500211 +- 0.103808


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
brain,89.1496 +- 21.9109,298.628 +- 58.686,534.995 +- 78.6223,1761.77 +- 188.454,19.2954 +- 1.44903,13.5041 +- 1.42003,0.808978 +- 0.0364361,0.500211 +- 0.103808


## Pelvis


In [ ]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cyclegan_pelvis_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [ ]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cyclegan_pelvis_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,81.8099 +- 23.8409,195.527 +- 88.7502
MSE,1443.98 +- 356.169,5180.7 +- 1113.42
PSNR,19.884 +- 1.91953,17.5001 +- 3.07259
SSIM,0.780998 +- 0.0178051,0.525097 +- 0.072071


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
pelvis,81.8099 +- 23.8409,195.527 +- 88.7502,1443.98 +- 356.169,5180.7 +- 1113.42,19.884 +- 1.91953,17.5001 +- 3.07259,0.780998 +- 0.0178051,0.525097 +- 0.072071


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,83.3963 +- 16.1116,208.604 +- 91.4367
MSE,1407.8 +- 297.994,5029.67 +- 969.391
PSNR,19.7102 +- 1.44886,17.0609 +- 3.10696
SSIM,0.780139 +- 0.0131045,0.529231 +- 0.0438917


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
pelvis,83.3963 +- 16.1116,208.604 +- 91.4367,1407.8 +- 297.994,5029.67 +- 969.391,19.7102 +- 1.44886,17.0609 +- 3.10696,0.780139 +- 0.0131045,0.529231 +- 0.0438917


## Thorax


In [ ]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cyclegan_thorax_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [ ]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cyclegan_thorax_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Slice metrics | non-finite values ignored in stats -> PSNR_masked: 3
Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,59.4869 +- 33.543,225.314 +- 125.209
MSE,836.03 +- 314.341,3115.16 +- 873.974
PSNR,23.2968 +- 6.85517,17.1849 +- 5.29755
SSIM,0.828421 +- 0.0499973,0.579952 +- 0.142547


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
TH,59.4869 +- 33.543,225.314 +- 125.209,836.03 +- 314.341,3115.16 +- 873.974,23.2968 +- 6.85517,17.1849 +- 5.29755,0.828421 +- 0.0499973,0.579952 +- 0.142547


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,57.9898 +- 21.9808,223.936 +- 91.9382
MSE,834.622 +- 199.369,3152.5 +- 396.074
PSNR,23.5396 +- 3.20496,16.9334 +- 3.144
SSIM,0.83001 +- 0.0358643,0.578733 +- 0.108016


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
TH,57.9898 +- 21.9808,223.936 +- 91.9382,834.622 +- 199.369,3152.5 +- 396.074,23.5396 +- 3.20496,16.9334 +- 3.144,0.83001 +- 0.0358643,0.578733 +- 0.108016


## Head and Neck


In [ ]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cyclegan_head_neck_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [ ]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cyclegan_head_neck_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,91.1659 +- 66.7375,226.898 +- 163.14
MSE,1178.66 +- 693.528,2758.63 +- 1080.99
PSNR,21.2734 +- 6.05697,17.2063 +- 5.86452
SSIM,0.755616 +- 0.0839252,0.55884 +- 0.128493


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
HN,91.1659 +- 66.7375,226.898 +- 163.14,1178.66 +- 693.528,2758.63 +- 1080.99,21.2734 +- 6.05697,17.2063 +- 5.86452,0.755616 +- 0.0839252,0.55884 +- 0.128493


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,96.6227 +- 37.5666,243.337 +- 102.645
MSE,1123.71 +- 273.516,2748.09 +- 425.621
PSNR,21.1913 +- 1.39603,17.0877 +- 1.5587
SSIM,0.752931 +- 0.0343386,0.543353 +- 0.0756491


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
HN,96.6227 +- 37.5666,243.337 +- 102.645,1123.71 +- 273.516,2748.09 +- 425.621,21.1913 +- 1.39603,17.0877 +- 1.5587,0.752931 +- 0.0343386,0.543353 +- 0.0756491


## Fullbody


In [ ]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cyclegan_allregions_final/test_50/test_metrics_over_all.csv")
df = load_metrics_csv(csv_path)

# show_volume_slice_influence(df, volume=3)


In [ ]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cyclegan_allregions_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Slice metrics | non-finite values ignored in stats -> PSNR_masked: 3
Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,84.8216 +- 47.7418,266.775 +- 140.064
MSE,914.259 +- 542.915,2949.94 +- 1439.7
PSNR,20.6946 +- 5.27471,15.5237 +- 4.54554
SSIM,0.792268 +- 0.0679484,0.538017 +- 0.138173


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,72.6371 +- 44.5113,236.676 +- 136.414,1197.06 +- 474.576,3913.5 +- 1217.07,22.3438 +- 5.77136,17.0925 +- 5.00594,0.799181 +- 0.0551391,0.511857 +- 0.123491
HN,104.959 +- 66.4038,258.368 +- 161.621,1127.21 +- 741.762,2597.74 +- 1163.81,20.2153 +- 6.2411,16.2041 +- 6.02258,0.735926 +- 0.0857004,0.530857 +- 0.125814
TH,72.7701 +- 38.6851,261.902 +- 142.37,921.577 +- 364.877,3340.38 +- 936.996,22.2851 +- 7.08415,16.4841 +- 5.46736,0.812795 +- 0.0559777,0.550422 +- 0.158763
brain,100.249 +- 38.1675,315.515 +- 116.943,468.342 +- 217.506,1631.17 +- 836.704,18.2935 +- 2.9337,13.066 +- 1.88121,0.792474 +- 0.0647163,0.52221 +- 0.150404
pelvis,62.6483 +- 33.4505,225.386 +- 130.929,1194.14 +- 398.007,4235.91 +- 1024.11,22.1514 +- 2.58131,16.6547 +- 2.66001,0.819159 +- 0.0380783,0.582515 +- 0.103954


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,83.4746 +- 32.7126,257.282 +- 89.6774
MSE,976.813 +- 362.318,3131.83 +- 1079.73
PSNR,21.1761 +- 3.03319,15.9352 +- 2.80049
SSIM,0.790805 +- 0.0454161,0.537648 +- 0.0850759


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,67.6774 +- 28.8377,223.947 +- 99.3627,1222.05 +- 303.29,4012.54 +- 554.366,22.9393 +- 3.47535,17.3856 +- 3.43723,0.804713 +- 0.0401958,0.522159 +- 0.0991575
HN,109.628 +- 35.9483,271.248 +- 95.8539,1069.56 +- 269.906,2608.42 +- 463.679,20.1939 +- 1.35968,16.1823 +- 1.38974,0.735768 +- 0.0332023,0.521134 +- 0.0598953
TH,70.4867 +- 29.4524,258.209 +- 110.93,929.237 +- 252.341,3435.85 +- 461.91,22.5959 +- 3.88221,16.3088 +- 3.58763,0.815293 +- 0.0417509,0.550936 +- 0.127377
brain,99.9188 +- 12.739,302.034 +- 28.8624,468.539 +- 73.8318,1520.28 +- 170.801,18.3097 +- 0.761675,13.1395 +- 0.563861,0.792574 +- 0.0182405,0.523556 +- 0.0481865
pelvis,63.8499 +- 18.0851,227.871 +- 67.8463,1174.07 +- 267.736,4198.39 +- 579.229,22.06 +- 1.90912,16.6049 +- 1.96479,0.817905 +- 0.0249799,0.574127 +- 0.0617434


# Experiment 2

## 32p99

### CUT

In [ ]:

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/2_experiment_cut_synthrad_abdomen_32p99/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)

Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,35.9848 +- 20.0743,117.439 +- 55.8068
MSE,14206 +- 18256.4,46868 +- 53915.9
PSNR,26.7446 +- 3.6136,21.748 +- 4.97039
SSIM,0.84099 +- 0.0456486,0.6273 +- 0.0656255


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,35.9848 +- 20.0743,117.439 +- 55.8068,14206 +- 18256.4,46868 +- 53915.9,26.7446 +- 3.6136,21.748 +- 4.97039,0.84099 +- 0.0456486,0.6273 +- 0.0656255


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,35.6425 +- 9.78007,117.371 +- 27.2922
MSE,13801.4 +- 7949.92,46183.3 +- 25364.5
PSNR,26.9311 +- 1.45544,21.6283 +- 1.77294
SSIM,0.840618 +- 0.0274701,0.627112 +- 0.0266859


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,35.6425 +- 9.78007,117.371 +- 27.2922,13801.4 +- 7949.92,46183.3 +- 25364.5,26.9311 +- 1.45544,21.6283 +- 1.77294,0.840618 +- 0.0274701,0.627112 +- 0.0266859


### CycleGAN

In [ ]:

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/2_experiment_cyclegan_abdomen_32p99/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)

Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,49.0058 +- 28.6093,161.627 +- 86.0609
MSE,25018 +- 24861.2,82621.8 +- 75541.1
PSNR,24.738 +- 5.16548,19.3559 +- 4.37438
SSIM,0.839393 +- 0.0413238,0.60744 +- 0.0851225


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,49.0058 +- 28.6093,161.627 +- 86.0609,25018 +- 24861.2,82621.8 +- 75541.1,24.738 +- 5.16548,19.3559 +- 4.37438,0.839393 +- 0.0413238,0.60744 +- 0.0851225


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,47.5474 +- 10.6656,159.636 +- 40.2211
MSE,23620.3 +- 9415.34,80242.7 +- 34676.9
PSNR,25.0519 +- 1.74163,19.3573 +- 1.89511
SSIM,0.840963 +- 0.0207205,0.607734 +- 0.0449712


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,47.5474 +- 10.6656,159.636 +- 40.2211,23620.3 +- 9415.34,80242.7 +- 34676.9,25.0519 +- 1.74163,19.3573 +- 1.89511,0.840963 +- 0.0207205,0.607734 +- 0.0449712


### Pix2pix

In [ ]:

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/2_experiment_pix2pix_synthrad_abdomen_32p99/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)



Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,39.0401 +- 20.0171,127.484 +- 59.2615
MSE,18245.5 +- 17965.6,58834.4 +- 53422.4
PSNR,25.6689 +- 4.68726,20.4688 +- 3.94256
SSIM,0.845541 +- 0.0375898,0.621406 +- 0.0877327


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,39.0401 +- 20.0171,127.484 +- 59.2615,18245.5 +- 17965.6,58834.4 +- 53422.4,25.6689 +- 4.68726,20.4688 +- 3.94256,0.845541 +- 0.0375898,0.621406 +- 0.0877327


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,38.4637 +- 8.33189,126.331 +- 31.2793
MSE,17707.9 +- 7235.62,57577.6 +- 26234.7
PSNR,25.8847 +- 1.64579,20.4744 +- 1.80046
SSIM,0.846378 +- 0.0186759,0.62222 +- 0.0473776


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,38.4637 +- 8.33189,126.331 +- 31.2793,17707.9 +- 7235.62,57577.6 +- 26234.7,25.8847 +- 1.64579,20.4744 +- 1.80046,0.846378 +- 0.0186759,0.62222 +- 0.0473776


In [ ]:

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/2_experiment_pix2pix_synthrad_abdomen_32p99/test_51/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)



Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,32.0179 +- 20.0824,104.004 +- 56.1877
MSE,13501.7 +- 17687.3,43306.4 +- 50704.8
PSNR,27.1599 +- 4.45923,21.9085 +- 3.76985
SSIM,0.868829 +- 0.0348851,0.676166 +- 0.0641248


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,32.0179 +- 20.0824,104.004 +- 56.1877,13501.7 +- 17687.3,43306.4 +- 50704.8,27.1599 +- 4.45923,21.9085 +- 3.76985,0.868829 +- 0.0348851,0.676166 +- 0.0641248


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,31.2946 +- 8.57739,103.022 +- 28.6145
MSE,12931.9 +- 7192.56,42504.9 +- 23820.9
PSNR,27.4138 +- 1.62499,21.936 +- 1.80846
SSIM,0.870179 +- 0.0196782,0.677075 +- 0.0374571


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,31.2946 +- 8.57739,103.022 +- 28.6145,12931.9 +- 7192.56,42504.9 +- 23820.9,27.4138 +- 1.62499,21.936 +- 1.80846,0.870179 +- 0.0196782,0.677075 +- 0.0374571


## Sep Input Layer

### CUT

In [ ]:

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/2_experiment_cut_synthrad_abdomen_sep_first_layer/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,58.7768 +- 83.2257,181.413 +- 217.377
MSE,1183.11 +- 478.082,3944.93 +- 1469.18
PSNR,25.2339 +- 5.2949,20.1274 +- 5.84495
SSIM,0.823941 +- 0.0762595,0.587645 +- 0.126163


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,58.7768 +- 83.2257,181.413 +- 217.377,1183.11 +- 478.082,3944.93 +- 1469.18,25.2339 +- 5.2949,20.1274 +- 5.84495,0.823941 +- 0.0762595,0.587645 +- 0.126163


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,63.4599 +- 84.5628,201.499 +- 241.396
MSE,1186.78 +- 376.854,3967.93 +- 1126.73
PSNR,25.1923 +- 4.07778,19.7329 +- 4.37479
SSIM,0.820426 +- 0.0756474,0.578012 +- 0.121299


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,63.4599 +- 84.5628,201.499 +- 241.396,1186.78 +- 376.854,3967.93 +- 1126.73,25.1923 +- 4.07778,19.7329 +- 4.37479,0.820426 +- 0.0756474,0.578012 +- 0.121299


### CycleGAN

In [ ]:

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/2_experiment_cyclegan_abdomen_sep_first_layer/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,72.9222 +- 51.6224,228.958 +- 153.319
MSE,1176.5 +- 532.625,3868.78 +- 1416.11
PSNR,22.1262 +- 5.70842,17.0692 +- 4.94052
SSIM,0.789977 +- 0.0565202,0.495445 +- 0.120165


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,72.9222 +- 51.6224,228.958 +- 153.319,1176.5 +- 532.625,3868.78 +- 1416.11,22.1262 +- 5.70842,17.0692 +- 4.94052,0.789977 +- 0.0565202,0.495445 +- 0.120165


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,68.9997 +- 44.7462,221.772 +- 145.263
MSE,1208.78 +- 434.606,3981.7 +- 1122.32
PSNR,22.7061 +- 3.60149,17.2915 +- 3.55574
SSIM,0.795345 +- 0.0452956,0.503543 +- 0.0999761


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,68.9997 +- 44.7462,221.772 +- 145.263,1208.78 +- 434.606,3981.7 +- 1122.32,22.7061 +- 3.60149,17.2915 +- 3.55574,0.795345 +- 0.0452956,0.503543 +- 0.0999761


### Pix2Pix

In [ ]:

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/2_experiment_pix2pix_synthrad_abdomen_sep_input_layers/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,36.564 +- 20.6881,119.717 +- 62.6723
MSE,952.284 +- 280.921,3165.85 +- 797.942
PSNR,25.6902 +- 3.29659,20.662 +- 3.32996
SSIM,0.852453 +- 0.0375695,0.635234 +- 0.0834602


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,36.564 +- 20.6881,119.717 +- 62.6723,952.284 +- 280.921,3165.85 +- 797.942,25.6902 +- 3.29659,20.662 +- 3.32996,0.852453 +- 0.0375695,0.635234 +- 0.0834602


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,35.5828 +- 9.59296,117.113 +- 34.9215
MSE,945.263 +- 148.325,3145.87 +- 346.612
PSNR,25.9438 +- 1.45772,20.7746 +- 1.83889
SSIM,0.854947 +- 0.0223107,0.639765 +- 0.0551154


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,35.5828 +- 9.59296,117.113 +- 34.9215,945.263 +- 148.325,3145.87 +- 346.612,25.9438 +- 1.45772,20.7746 +- 1.83889,0.854947 +- 0.0223107,0.639765 +- 0.0551154


## Nyul

### CUT

In [ ]:

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/2_experiment_cut_synthrad_abdomen_33nyul/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,40.5094 +- 21.8186,132.333 +- 66.3542
MSE,1310.64 +- 459.891,4241.47 +- 1335.41
PSNR,25.5474 +- 3.19162,20.8576 +- 6.2989
SSIM,0.822461 +- 0.0704862,0.591744 +- 0.0743775


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,40.5094 +- 21.8186,132.333 +- 66.3542,1310.64 +- 459.891,4241.47 +- 1335.41,25.5474 +- 3.19162,20.8576 +- 6.2989,0.822461 +- 0.0704862,0.591744 +- 0.0743775


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,40.3677 +- 8.91578,132.474 +- 24.9229
MSE,1341.41 +- 332.127,4359.16 +- 608.45
PSNR,25.6496 +- 1.32207,20.6097 +- 1.78157
SSIM,0.820755 +- 0.0305097,0.592906 +- 0.0335432


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,40.3677 +- 8.91578,132.474 +- 24.9229,1341.41 +- 332.127,4359.16 +- 608.45,25.6496 +- 1.32207,20.6097 +- 1.78157,0.820755 +- 0.0305097,0.592906 +- 0.0335432


### CycleGAN

In [ ]:

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/2_experiment_cyclegan_abdomen_33nyul/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,51.1053 +- 25.5954,169.985 +- 77.3747
MSE,1366.46 +- 494.888,4516.21 +- 1318.34
PSNR,23.9676 +- 4.96924,18.5612 +- 4.12271
SSIM,0.825471 +- 0.038415,0.567514 +- 0.0810917


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,51.1053 +- 25.5954,169.985 +- 77.3747,1366.46 +- 494.888,4516.21 +- 1318.34,23.9676 +- 4.96924,18.5612 +- 4.12271,0.825471 +- 0.038415,0.567514 +- 0.0810917


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,50.0852 +- 8.54488,168.811 +- 34.3632
MSE,1381 +- 302.852,4584.63 +- 602.099
PSNR,24.1774 +- 1.38157,18.4725 +- 1.43905
SSIM,0.825856 +- 0.0180467,0.565184 +- 0.0417491


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,50.0852 +- 8.54488,168.811 +- 34.3632,1381 +- 302.852,4584.63 +- 602.099,24.1774 +- 1.38157,18.4725 +- 1.43905,0.825856 +- 0.0180467,0.565184 +- 0.0417491


### Pix2Pix

In [ ]:

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/2_experiment_pix2pix_synthrad_abdomen_33nyul/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,32.6277 +- 19.8734,107.784 +- 56.7526
MSE,902.302 +- 266.583,3010.36 +- 740.653
PSNR,26.929 +- 4.57145,21.5935 +- 3.80485
SSIM,0.865392 +- 0.0368798,0.667136 +- 0.0739568


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,32.6277 +- 19.8734,107.784 +- 56.7526,902.302 +- 266.583,3010.36 +- 740.653,26.929 +- 4.57145,21.5935 +- 3.80485,0.865392 +- 0.0368798,0.667136 +- 0.0739568


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,31.972 +- 8.58828,106.765 +- 29.2104
MSE,899.776 +- 144.677,3012.53 +- 366.223
PSNR,27.1635 +- 1.88795,21.6021 +- 1.87593
SSIM,0.866814 +- 0.0201792,0.668818 +- 0.0441601


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,31.972 +- 8.58828,106.765 +- 29.2104,899.776 +- 144.677,3012.53 +- 366.223,27.1635 +- 1.88795,21.6021 +- 1.87593,0.866814 +- 0.0201792,0.668818 +- 0.0441601


## N-Peaks

In [ ]:
base_results_dir = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results")

cut_npeaks_models = [
    "2_experiment_cut_synthrad_abdomen_34normalized_n4_03LIC",
    "2_experiment_cut_synthrad_abdomen_34normalized_n4_08LIC",
    "2_experiment_cut_synthrad_abdomen_34normalized_n4_centerspecific_03LIC",
    "2_experiment_cut_synthrad_abdomen_34normalized_n4_centerspecific_08LIC",
]

for model_name in cut_npeaks_models:
    print(f"\n=== {model_name} ===")
    csv_path = base_results_dir / model_name / "test_3" / "test_metrics_over_all.csv"
    show_metrics_for_csv(csv_path)



=== 2_experiment_cut_synthrad_abdomen_34normalized_n4_03LIC ===
Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,37.1961 +- 20.2649,120.447 +- 54.1621
MSE,14216 +- 18113.5,46426.2 +- 50417.4
PSNR,26.5967 +- 3.26947,22.1034 +- 7.513
SSIM,0.838883 +- 0.0603891,0.626433 +- 0.0687307


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,37.1961 +- 20.2649,120.447 +- 54.1621,14216 +- 18113.5,46426.2 +- 50417.4,26.5967 +- 3.26947,22.1034 +- 7.513,0.838883 +- 0.0603891,0.626433 +- 0.0687307


Overall metrics (volume-based):


...

,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,36.4517 +- 22.0294,117.665 +- 59.5966,14440.7 +- 19516.3,47233.8 +- 55892.7,26.7398 +- 3.39104,21.1149 +- 2.54718,0.841138 +- 0.0652494,0.633856 +- 0.0698261


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,36.612 +- 13.3057,119.574 +- 36.7395
MSE,14387.6 +- 10748.1,47944.6 +- 33792.5
PSNR,26.8267 +- 1.60001,21.1643 +- 1.69287
SSIM,0.839314 +- 0.0368338,0.633317 +- 0.0331764


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,36.612 +- 13.3057,119.574 +- 36.7395,14387.6 +- 10748.1,47944.6 +- 33792.5,26.8267 +- 1.60001,21.1643 +- 1.69287,0.839314 +- 0.0368338,0.633317 +- 0.0331764


In [13]:
base_results_dir = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results")

cyclegan_npeaks_models = [
    "2_experiment_cyclegan_abdomen_34normalized_n4_03LIC",
    "2_experiment_cyclegan_abdomen_34normalized_n4_08LIC",
    "2_experiment_cyclegan_abdomen_34normalized_n4_centerspecific_03LIC",
    "2_experiment_cyclegan_abdomen_34normalized_n4_centerspecific_08LIC",
]

for model_name in cyclegan_npeaks_models:
    print(f"\n=== {model_name} ===")
    csv_path = base_results_dir / model_name / "test_50" / "test_metrics_over_all.csv"
    show_metrics_for_csv(csv_path)



=== 2_experiment_cyclegan_abdomen_34normalized_n4_03LIC ===
Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,43.4045 +- 22.1615,141.955 +- 62.4244
MSE,20383.5 +- 20100.6,66876.9 +- 58640.9
PSNR,25.1933 +- 4.49435,19.9016 +- 4.05259
SSIM,0.832866 +- 0.0367741,0.60398 +- 0.0697146


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,43.4045 +- 22.1615,141.955 +- 62.4244,20383.5 +- 20100.6,66876.9 +- 58640.9,25.1933 +- 4.49435,19.9016 +- 4.05259,0.832866 +- 0.0367741,0.60398 +- 0.0697146


Overall metrics (volume-based):


...

,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,58.2927 +- 30.5144,193.853 +- 93.6005,32963.7 +- 27244.8,109848 +- 84289.8,23.2551 +- 5.14926,17.8528 +- 4.31583,0.813857 +- 0.0416644,0.535025 +- 0.0869236


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,56.7758 +- 11.0821,191.986 +- 46.0324
MSE,31478.3 +- 10260.1,107480 +- 39909
PSNR,23.519 +- 1.51765,17.788 +- 1.6825
SSIM,0.81488 +- 0.0209007,0.533365 +- 0.0450953


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,56.7758 +- 11.0821,191.986 +- 46.0324,31478.3 +- 10260.1,107480 +- 39909,23.519 +- 1.51765,17.788 +- 1.6825,0.81488 +- 0.0209007,0.533365 +- 0.0450953


In [14]:
base_results_dir = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results")

pix2pix_npeaks_models = [
    "2_experiment_pix2pix_synthrad_abdomen_34normalized_n4_03LIC",
    "2_experiment_pix2pix_synthrad_abdomen_34normalized_n4_08LIC",
    "2_experiment_pix2pix_synthrad_abdomen_34normalized_n4_centerspecific_03LIC",
    "2_experiment_pix2pix_synthrad_abdomen_34normalized_n4_centerspecific_08LIC",
]

for model_name in pix2pix_npeaks_models:
    print(f"\n=== {model_name} ===")
    csv_path = base_results_dir / model_name / "test_50" / "test_metrics_over_all.csv"
    show_metrics_for_csv(csv_path)



=== 2_experiment_pix2pix_synthrad_abdomen_34normalized_n4_03LIC ===
Overall metrics (slice-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,32.4495 +- 19.4722,107.095 +- 54.7119
MSE,13951.8 +- 17470,45564.1 +- 50081.6
PSNR,26.9642 +- 4.62713,21.5902 +- 3.74425
SSIM,0.863691 +- 0.0356726,0.66171 +- 0.0691067


Metrics by body part (slice-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,32.4495 +- 19.4722,107.095 +- 54.7119,13951.8 +- 17470,45564.1 +- 50081.6,26.9642 +- 4.62713,21.5902 +- 3.74425,0.863691 +- 0.0356726,0.66171 +- 0.0691067


Overall metrics (volume-based):


...

,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,33.5813 +- 20.4381,109.456 +- 56.5745,14541.7 +- 18195,46751.9 +- 51887.2,26.8133 +- 4.65127,21.4913 +- 3.74742,0.862277 +- 0.0365666,0.659603 +- 0.0668545


Overall metrics (volume-based):


,unmasked (mean +- std),masked (mean +- std)
metric,,
MAE,32.886 +- 9.08875,108.73 +- 29.1889
MSE,13964.8 +- 7687.47,46049.9 +- 25116.9
PSNR,27.0597 +- 1.59211,21.4815 +- 1.70183
SSIM,0.863726 +- 0.0215314,0.660845 +- 0.038587


Metrics by body part (volume-based):


,MAE_unmasked,MAE_masked,MSE_unmasked,MSE_masked,PSNR_unmasked,PSNR_masked,SSIM_unmasked,SSIM_masked
body_part,,,,,,,,
AB,32.886 +- 9.08875,108.73 +- 29.1889,13964.8 +- 7687.47,46049.9 +- 25116.9,27.0597 +- 1.59211,21.4815 +- 1.70183,0.863726 +- 0.0215314,0.660845 +- 0.038587
